# Response Generation: Parameters and Structured Outputs

In this session, we will examine the mechanisms used to control the output of Large Language Models (LLMs). Generating a response is not a simple text-retrieval process; it is a probabilistic calculation where the model predicts the next token in a sequence.

As developers, we can influence this calculation using specific generation parameters. We will cover:
1.  **Stochasticity Control:** Understanding `temperature` and `top_p`.
2.  **Output Constraints:** Managing `max_tokens` and `stop_sequences`.
3.  **Data Integrity:** Implementing Structured Outputs (JSON mode) for programmatic use.

#### Environment Setup

In [22]:
import os
import json
from dotenv import load_dotenv
from google import genai
from huggingface_hub import InferenceClient

load_dotenv(override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
HF_TOKEN = os.getenv("HF_TOKEN")

if GOOGLE_API_KEY and HF_TOKEN:
    print("Check: Keys Loaded successfully.")
else:
    print("Check: Keys Missing! Check your `.env` file.")

Check: Keys Loaded successfully.


In [23]:
# Initialise the Cloud Client
client = genai.Client(api_key=GOOGLE_API_KEY)

### 1. Stochasticity Control

#### Temperature

The `temperature` parameter scales the probability distribution of the model's next-token predictions. 

*   **Low Temperature (e.g., 0.0):** The model becomes **deterministic**, consistently choosing the most probable token. This is ideal for factual queries, coding, and mathematical logic.
*   **High Temperature (e.g., 1.0+):** The model introduces more variance, allowing for "less likely" tokens to be selected.

In [24]:
def test_temperature_variance(target_temp, model_id="gemini-3.5-flash-lite"):
    prompt = "Write a 30 word thank you email to a customer."
    results = []
    
    response = client.models.generate_content(
        model=model_id,
        contents=prompt,
        config={"temperature": target_temp}
    )
    results.append(response.text.strip())
    
    return results

# Experiment 1: Deterministic (Temperature 0.0)
print(f"Results at Temp 0.0: {test_temperature_variance(0.0)}")

# Experiment 3: Creative (Temperature 2)
print(f"Results at Temp 2: {test_temperature_variance(2)}")

Results at Temp 0.0: ['Subject: Thank you!\n\nHi [Name],\n\nThank you so much for your recent purchase! We truly appreciate your support and hope you love your new [Product]. Please reach out if you need anything. \n\nBest,\n[Your Name]']
Results at Temp 2: ['Subject: Thank you!\n\nHi [Name],\n\nThank you so much for your recent purchase! We truly appreciate your support and hope you love your new [Product]. Please reach out if you need anything. \n\nBest,\n[Your Name]']


#### Sampling and Filtering

Beyond `temperature`, probability distributions can be refined using nucleus sampling (`top_p`) and top-k filtering (`top_k`).

**`top_p` (Nucleus Sampling)**
* Restricts sampling to the smallest set of tokens whose cumulative probability exceeds `p`
* Range: 0.0 to 1.0
* Balances diversity and coherence. Prevents low-probability 'outlier' tokens whilst permitting controlled variance
* Example: `top_p=0.9` includes tokens until 90% of probability mass is covered

**`top_k` Filtering**
* Limits token selection to the `k` most probable tokens.
* Range: 1 to vocabulary size
* Simple, effective control for reducing nonsensical outputs.
* Example: `top_k=50` means only the 50 most probable tokens are considered.


*When both are specified, `top_k` is applied first, then `top_p` filters the remaining tokens*

### 2. Managing Output Length and Termination

To ensure cost-efficiency and prevent the model from generating unnecessary text, we use `max_output_tokens` and `stop_sequences`.

*   `max_output_tokens`: Sets a hard limit on the number of tokens generated. If the limit is reached, the model terminates the response immediately, even if it is mid-sentence.
*  `stop_sequences`: Instructs the model to stop generating as soon as a specific character or word is encountered. This is useful for preventing the model from rambling or for stopping a list at a specific point.


In [25]:
def demonstrate_limits(MODEL_ID="gemini-3.5-flash-lite"):
    """
    Shows the effect of token limits and stop sequences on model output.
    """
    prompt = "Write a three-step instruction for making tea."
    
    # Example 1: Token Limit (Truncated Output)
    print("--- Truncated Response (Limit: 10 Tokens) ---")
    truncated = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={"max_output_tokens": 10}
    )
    print(truncated.text)

    # Example 2: Stop Sequence (Controlled Exit)
    # We tell the model to stop if it attempts to write 'Step 2'
    print("\n--- Controlled Response (Stop at 'Step 2') ---")
    stopped = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={"stop_sequences": ["2"]}
    )
    print(stopped.text)

demonstrate_limits()

--- Truncated Response (Limit: 10 Tokens) ---
1. Boil fresh water and

--- Controlled Response (Stop at 'Step 2') ---
1. Place a tea bag into your favorite mug and pour freshly boiled water over it. 



### 3. Structured Outputs (JSON Mode)

In a professional software environment, raw text is often difficult to parse into application logic. Developers require **Structured Data**, typically in JSON format.

Rather than relying purely on prompt engineering (e.g., "Please return JSON"), the Gemini API allows us to enforce a strict schema using the `response_mime_type` parameter. This ensures the output is a valid JSON object that can be converted directly into a Python dictionary.

In [26]:
def extract_structured_data(raw_text, MODEL_ID="gemini-3.5-flash-lite"):
    """
    Parses unstructured text into a structured JSON object.
    """
    prompt = f"""
    Extract the following entities from the text: 'name', 'age', and 'profession'.
    Text: "{raw_text}"
    """
    
    # Enforcing JSON output via the config parameter
    response = client.models.generate_content(
        model=MODEL_ID,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "temperature": 0.1
        }
    )
    
    # Parse the string output into a Python dictionary
    return json.loads(response.text)

# Example Execution
user_bio = "Hello, I am David. I am a 34-year-old architect living in Berlin."
structured_result = extract_structured_data(user_bio)

print("Type of result:", type(structured_result))
print("JSON Output:", structured_result)

Type of result: <class 'dict'>
JSON Output: {'name': 'David', 'age': 34, 'profession': 'architect'}


However, to ensure that the response exactly follows a predefined structure, we can also create a schema as needed.

### 4. Implementing a Minimal Response Generator

We can now synthesise these concepts into a single, robust function. A **Minimal Response Generator** should handle the system instructions, the user prompt, and the relevant generation parameters while providing basic error handling.

This function represents a production-ready wrapper that can be integrated into larger Python applications.


In [27]:
def generate_response(
    user_query, 
    system_role="You are a helpful assistant.", 
    temp=0.7, 
    max_len=200, 
    json_mode=False,
    MODEL_ID="gemini-3.5-flash-lite"
):
    """
    A synthesized generator function incorporating role, parameters, and format control.
    """
    config = {
        "temperature": temp,
        "max_output_tokens": max_len
    }
    
    if json_mode:
        config["response_mime_type"] = "application/json"
        
    # Assembling the payload
    # Note: We provide the system role as the first part of the context
    contents = [f"SYSTEM INSTRUCTION: {system_role}", f"USER QUERY: {user_query}"]
    
    try:
        response = client.models.generate_content(
            model=MODEL_ID,
            contents=contents,
            config=config
        )
        return response.text
    except Exception as e:
        return f"System Error: {str(e)}"

# Implementation Test: Generating a List
result = generate_response(
    user_query="Provide a list of 3 primary colors.",
    system_role="You are a concise design assistant. Provide answers in a bulleted list.",
    temp=0.0
)

print(result)

* Red
* Blue
* Yellow


### 5. System Instructions (Role-Based Context)

System instructions define the persona and constraints for the model, providing persistent context for multi-turn conversations. System instructions are set once and applied to all subsequent messages. These instructions take precedence over user prompts in cases of conflict.


Use Cases
1. Roles: 'You are a data science tutor.'
2. Constraint Enforcement: 'Respond in exactly three sentences.'
3. Domain Expertise: 'You specialise in machine learning.'
4. Consistency: Maintaining tone and style across multiple API calls.

In [29]:
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)

def system_instructions():
    user_query = 'Explain neural networks.'
    system_instructions = [
        'You are a data science educator. Explain concepts pedagogically, using analogies.',
        'You are a technical consultant. Use formal terminology and assume advanced knowledge.'
    ]
    
    for i, system_role in enumerate(system_instructions, 1):
        print(f'\n--- System Role {i} ---')
        response = gemini_client.models.generate_content(
            model="gemini-3.5-flash-lite",
            contents=[f'SYSTEM: {system_role}', f'USER: {user_query}'],
            config={'temperature': 0.7, 'max_output_tokens': 100}
        )
        print(response.text[:150])

system_instructions()


--- System Role 1 ---
Welcome to data science! Today, we’re going to unlock the mystery of **Neural Networks**—the computational engines behind everything from self-driving

--- System Role 2 ---
Artificial Neural Networks (ANNs) constitute a paradigm of machine learning computationally modeled after biological neural architectures. They are pa


## Parameters Summary

Let's summarise the various parameters available for Gemini and Llama

**Stochasticity and Sampling**
* `temperature`: Base randomness. 0.0 = deterministic, 1.0+ = creative
* `top_p`: Nucleus sampling—restricts to tokens covering `p` fraction of probability
* `top_k`: Token filtering—considers only top-k most probable tokens

**Output Control**
* `max_output_tokens`: Hard upper limit on response length
* `min_output_tokens`: Enforces minimum response length (Gemini)
* `stop_sequences`: Terminates generation at specific strings

**Penalty Mechanisms (Llama)**
* `frequency_penalty`: Discourages token repetition
* `presence_penalty`: Encourages vocabulary diversity

**Reproducibility and Context**
* `seed`: Enables deterministic output for testing
* System instructions: Define persistent role and constraints

These are just a few common parameters that may help in getting the response you want. To get a better understanding, it is recommended to visit the [official Gemini](https://ai.google.dev/api/generate-content) and [HuggingFace documentation](https://huggingface.co/docs/huggingface_hub/v0.22.2/en/package_reference/inference_client#inference)